# GNN Robustness — Adapted-NETTACK only, no surrogate cache (Colab, single dataset)

Runs the **adapted NETTACK** evasion attack against trained GNN checkpoints on
one dataset (`elliptic` or `ellipticpp_actors`), then aggregates the results
into a paper-ready CSV plus plots / LaTeX tables.

This notebook is a specialized version of
`GNN_Robustness_FGSM_PGD_NodeInjection_Nettack_Tdgia.ipynb`. It keeps the same
Colab setup, Drive wiring, dataset caching, checkpoint checks, artifact layout,
and summary logic, but removes the non-NETTACK attack grids.

Models supported: `gcn`, `gat`, `graphsage`, `chronowave_gnn` (static) and
`recgnn`, `evolvegcn_o`, `cosemignn` (temporal).

**Important adapted-NETTACK behavior**
- `ATTACK_FRACTION` is exposed as an experiment hyperparameter and is included
  in the sweep output / summaries.
- The adapted-NETTACK surrogate is **not cached**. Each sweep run trains its own
  linearized-GCN surrogate with the current run settings.
- Temporal drivers still train one shared surrogate on the union of train
  timesteps inside each run, then reuse it across that run's test slices.

**Setup assumptions**
- The repo is cloned from GitHub into `/content/<REPO_DIRNAME>`.
- Raw data lives in your **Google Drive** at `My Drive/data/` and mirrors the
  repo `data/` layout (i.e. `data/raw/elliptic/*.csv` and
  `data/raw/ellipticpp/actors/*.csv`). It is symlinked into the repo's `data/`.
- Trained checkpoints live in your **Google Drive** at `My Drive/models/` and
  mirror the repo `models/` layout.


## Install PyTorch Geometric and PyTorch Geometric Temporal

In [ ]:
# ============================================================
# Colab setup: PyTorch Geometric + PyTorch Geometric Temporal
# ============================================================

import sys
import subprocess
import torch

def pip_install(args):
    print("$ " + " ".join(map(str, [sys.executable, "-m", "pip", "install", *args])))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *args])

print("torch =", torch.__version__, "| cuda =", torch.version.cuda)

# For your current Colab:
# torch = 2.10.0+cu128
# cuda = 12.8
PYG_WHEEL_URL = "https://data.pyg.org/whl/torch-2.10.0+cu128.html"

# ------------------------------------------------------------
# 1. Install PyG compiled dependencies
# ------------------------------------------------------------
try:
    import torch_scatter
    import torch_sparse
    import pyg_lib
    print("PyG compiled dependencies already installed.")
except Exception:
    pip_install([
        "pyg_lib",
        "torch_scatter",
        "torch_sparse",
        "-f",
        PYG_WHEEL_URL
    ])

# ------------------------------------------------------------
# 2. Install torch-geometric
# ------------------------------------------------------------
try:
    import torch_geometric
    print("torch_geometric already installed:", torch_geometric.__version__)
except Exception:
    pip_install(["torch-geometric"])

# ------------------------------------------------------------
# 3. Install torch-geometric-temporal
# ------------------------------------------------------------
try:
    import torch_geometric_temporal
    print("torch_geometric_temporal already installed.")
except Exception:
    pip_install(["torch-geometric-temporal", "--no-deps"])

# ------------------------------------------------------------
# 4. Install pyyaml, used by run_sweep.py
# ------------------------------------------------------------
try:
    import yaml
    print("pyyaml already installed.")
except Exception:
    pip_install(["-U", "pyyaml"])

print("\nSetup finished.")
print("If this is the first time you installed these packages in this session, click:")
print("Runtime → Restart session")

# Restart the session: Runtime -> Restart Session

In [ ]:
# ------------------------------------------------------------
# Test cell
# ------------------------------------------------------------

import torch
import torch_geometric
import torch_geometric_temporal
import yaml

print("torch:", torch.__version__)
print("torch_geometric:", torch_geometric.__version__)
print("torch_geometric_temporal works")
print("pyyaml works")

## 0) Experiment configuration

In [ ]:
#@title Experiment configuration

# ---- Repo (GitHub clone) ----
REPO_URL     = "https://github.com/koshimbetovv/gnn-robustness-blockchain-research.git"  #@param {type:"string"}
REPO_BRANCH  = "main"                                                                     #@param {type:"string"}
REPO_DIRNAME = "gnn-robustness-blockchain-research"                                        #@param {type:"string"}

# ---- Google Drive data location ----
# Mounts My Drive at /content/drive and expects the data tree at:
#   /content/drive/MyDrive/data/raw/elliptic/...
#   /content/drive/MyDrive/data/raw/ellipticpp/actors/...
DRIVE_DATA_REL = "data"  #@param {type:"string"}

# ---- Adapted-NETTACK only: pick one dataset for this run ----
ATTACK_TYPE = "adapted_nettack"
DATASET     = "ellipticpp_actors"    #@param ["elliptic", "ellipticpp_actors"]

# Internal: helper code below works in terms of single-element lists,
# matching what scripts/run_sweep.py expects.
DATASETS = [DATASET]
ATTACKS  = [ATTACK_TYPE]

# ---- Models to attack (must already have checkpoints under models/Elliptic[++]) ----
MODELS = [
    "gcn", "gat", "graphsage", "chronowave_gnn",
    "recgnn", "evolvegcn_o", "cosemignn",
]

# ---- Attack-selection randomness ----
ATTACK_SEEDS = [0, 1, 2]  #@param {type:"raw"}

# ---- Target-selection knobs ----
SPLIT = "test"
ATTACK_ONLY_ILLICIT = True   #@param {type:"boolean"}
ONLY_CLEAN_CORRECT = True    #@param {type:"boolean"}

# Adapted-NETTACK target fraction hyperparameter.
# Use a list to sweep multiple values, e.g. [0.25, 0.5, 1.0].
ATTACK_FRACTIONS = [1.0]     #@param {type:"raw"}
ATTACK_FRACTION = float(ATTACK_FRACTIONS[0])

# ---- Output ----
ARTIFACTS_DIR = "artifacts_adapted_nettack_no_surrogate_cache"

# ---- Raw data file expectations (relative to the repo root) ----
ELLIPTIC_RAW_DIR = "data/raw/elliptic"
ELLIPTIC_REQUIRED_FILES = [
    "elliptic_txs_features.csv",
    "elliptic_txs_classes.csv",
    "elliptic_txs_edgelist.csv",
]
ELLIPTICPP_RAW_DIR = "data/raw/ellipticpp/actors"
ELLIPTICPP_REQUIRED_FILES = [
    "wallets_features.csv",
    "wallets_classes.csv",
    "AddrAddr_edgelist.csv",
]

print(f"Will run attack={ATTACK_TYPE!r} on dataset={DATASET!r}.")
print(f"ATTACK_FRACTIONS={ATTACK_FRACTIONS}")


## 1) Mount Google Drive and clone the repo

In [ ]:
import os, sys, json, shutil, subprocess, time
from pathlib import Path

def run(cmd, cwd=None):
    print("$ " + " ".join(map(str, cmd)))
    subprocess.check_call(list(map(str, cmd)), cwd=cwd)

# --- Mount Drive ---
from google.colab import drive  # type: ignore
drive.mount("/content/drive", force_remount=False)

DRIVE_DATA_ABS = Path("/content/drive/MyDrive") / DRIVE_DATA_REL
assert DRIVE_DATA_ABS.exists(), (
    f"Expected Drive data folder at {DRIVE_DATA_ABS}. "
    f"Place your data tree (raw/elliptic, raw/ellipticpp/actors, ...) there."
)
print("Drive data:", DRIVE_DATA_ABS)

# --- Clone the repo into /content ---
WORK = Path("/content")
REPO_ROOT = (WORK / REPO_DIRNAME).resolve()
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])

print("REPO_ROOT =", REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))


## 2) Wire the repo's `data/` directory to Google Drive

The repo's loaders read raw CSVs from `<REPO_ROOT>/data/...`. We replace that
folder with a symlink to your Drive `data/` folder so loads happen straight
from Drive (no copies, no re-uploads).


In [ ]:
repo_data = REPO_ROOT / "data"
if repo_data.is_symlink() or repo_data.exists():
    if repo_data.is_symlink():
        repo_data.unlink()
    else:
        shutil.rmtree(repo_data)

repo_data.symlink_to(DRIVE_DATA_ABS, target_is_directory=True)
print(f"Linked {repo_data} -> {DRIVE_DATA_ABS}")

# Sanity check: required files visible through the symlink
def check(rel_dir, required, label):
    rd = REPO_ROOT / rel_dir
    missing = [f for f in required if not (rd / f).exists()]
    if missing:
        raise FileNotFoundError(
            f"{label}: missing files {missing} under {rd}. "
            f"Put them in your Drive at {DRIVE_DATA_ABS}/{rel_dir.split('data/',1)[-1]}/"
        )
    print(f"✓ {label}: all raw files present at {rd}")

if "elliptic" in DATASETS:
    check(ELLIPTIC_RAW_DIR, ELLIPTIC_REQUIRED_FILES, "Elliptic")
if "ellipticpp_actors" in DATASETS:
    check(ELLIPTICPP_RAW_DIR, ELLIPTICPP_REQUIRED_FILES, "Elliptic++ actors")


## 3) Wire the repo's `models/` directory to Google Drive

Attacks need pretrained checkpoints from `models/Elliptic/` and/or `models/Elliptic++`.
We replace the repo's `models/` folder with a symlink to your Drive `models/` folder so checkpoints are loaded directly from Drive.

In [ ]:
DRIVE_MODELS_ABS = Path("/content/drive/MyDrive") / "models"
assert DRIVE_MODELS_ABS.exists(), (
    f"Expected Drive models folder at {DRIVE_MODELS_ABS}. "
    f"Please create a 'models' folder in your Google Drive and place your trained model checkpoints there."
)
print("Drive models:", DRIVE_MODELS_ABS)

repo_models = REPO_ROOT / "models"
if repo_models.is_symlink() or repo_models.exists():
    if repo_models.is_symlink():
        repo_models.unlink()
    else:
        shutil.rmtree(repo_models)

repo_models.symlink_to(DRIVE_MODELS_ABS, target_is_directory=True)
print(f"Linked {repo_models} -> {DRIVE_MODELS_ABS}")

## 3b) Cache dataset loaders for the in-notebook sweep

Each attack run normally re-instantiates the dataset (re-reads CSVs, rebuilds
PyG `Data` / temporal sequences). For a single-attack sweep with many
hyperparameter combinations this dominates wall time. The cell below
monkey-patches the dataset classes / loader functions so the heavy load happens
**once per (class, config)** and subsequent calls return the cached object.

We also make `build_paper_features` (the ChronoWaveGNN feature wavelet stack)
idempotent — without this, calling it twice on the same cached `Data` would
double-stack features.

Because we now run `scripts/run_sweep.py` in-process (next cell), these
monkey-patches are picked up by the attack drivers it imports.


In [ ]:
# Memoize dataset loaders so identical (class, cfg) calls return the cached
# object. Reduces a sweep's dataset-loading cost from O(combinations) to O(1).

import sys
sys.path.insert(0, str(REPO_ROOT))

_DATASET_CACHE = {}

def _patch_dataset_class(cls, get_method_name):
    if getattr(cls, "_cache_patched", False):
        return
    orig_init = cls.__init__
    orig_get  = getattr(cls, get_method_name)

    def cached_init(self, cfg=None, *args, **kwargs):
        try:
            key = (cls.__name__, repr(cfg), repr(args), repr(sorted(kwargs.items())))
        except Exception:
            key = (cls.__name__, id(cfg))
        self._cache_key = key
        self.cfg = cfg
        if key in _DATASET_CACHE:
            return
        orig_init(self, cfg, *args, **kwargs)

    def cached_get(self, *args, **kwargs):
        if self._cache_key not in _DATASET_CACHE:
            _DATASET_CACHE[self._cache_key] = orig_get(self, *args, **kwargs)
        return _DATASET_CACHE[self._cache_key]

    cls.__init__ = cached_init
    setattr(cls, get_method_name, cached_get)
    cls._cache_patched = True

# Static-graph datasets
from src.datasets.elliptic import EllipticDataset
from src.datasets.ellipticpp_actors import EllipticPPActorsDataset
_patch_dataset_class(EllipticDataset, "get_data")
_patch_dataset_class(EllipticPPActorsDataset, "get_data")

# RecGNN sequences
from src.datasets.recgnn_elliptic import RecGNNEllipticDataset
from src.datasets.recgnn_ellipticpp_actors import RecGNNEllipticPPActorsDataset
_patch_dataset_class(RecGNNEllipticDataset, "get_sequence")
_patch_dataset_class(RecGNNEllipticPPActorsDataset, "get_sequence")

# EvolveGCN sequences
from src.datasets.evolvegcn_elliptic import EvolveGCNEllipticDataset
from src.datasets.evolvegcn_ellipticpp_actors import EvolveGCNEllipticPPActorsDataset
_patch_dataset_class(EvolveGCNEllipticDataset, "get_sequence")
_patch_dataset_class(EvolveGCNEllipticPPActorsDataset, "get_sequence")

# CoSemiGNN loaders are module-level functions; memoize by their file-path args.
import functools
from src.datasets import cosemignn_elliptic as _cmod_e
from src.datasets import cosemignn_ellipticpp_addraddr as _cmod_ep

def _memo_loader(mod, fname):
    if getattr(mod, fname + "_orig", None) is not None:
        return
    orig = getattr(mod, fname)
    setattr(mod, fname + "_orig", orig)
    cache = {}
    @functools.wraps(orig)
    def wrapper(*args, **kwargs):
        key = (
            kwargs.get("feature_path"),
            kwargs.get("class_path"),
            kwargs.get("edge_path"),
            kwargs.get("semi_cache_dir"),
            bool(kwargs.get("rebuild_semi", False)),
        )
        if key not in cache:
            cache[key] = orig(*args, **kwargs)
        return cache[key]
    setattr(mod, fname, wrapper)

_memo_loader(_cmod_e, "load_cosemignn_elliptic")
_memo_loader(_cmod_ep, "load_cosemignn_ellipticpp_addraddr")

# Make ChronoWaveGNN's feature builder idempotent so reusing the cached `Data`
# across runs does not re-stack the wavelet features.
from src.datasets import chronowave_features as _cwf
if not getattr(_cwf, "_idempotent_patched", False):
    _orig_bpf = _cwf.build_paper_features
    def _idempotent_build_paper_features(data):
        if getattr(data, "raw_feature_dim", None) is not None:
            return  # already built on this cached Data
        return _orig_bpf(data)
    _cwf.build_paper_features = _idempotent_build_paper_features
    _cwf._idempotent_patched = True

print("Dataset loaders memoized for this notebook process.")
print("Patched classes:",
      [EllipticDataset.__name__, EllipticPPActorsDataset.__name__,
       RecGNNEllipticDataset.__name__, RecGNNEllipticPPActorsDataset.__name__,
       EvolveGCNEllipticDataset.__name__, EvolveGCNEllipticPPActorsDataset.__name__])


## 4) Verify trained checkpoints

Attacks need pretrained checkpoints in `models/Elliptic/` and/or `models/Elliptic++/`.
This notebook does not train — use the per-model `notebooks/colab_training/*.ipynb`
notebooks to produce checkpoints if needed. Checkpoints are committed to the
repo (LFS or otherwise), so cloning above should already provide them.


In [ ]:
DATASET_TO_MODELDIR = {
    "elliptic": "models/Elliptic",
    "ellipticpp_actors": "models/Elliptic++",
}

def latest_ckpt(model_name: str, model_dir_rel: str):
    md_abs = REPO_ROOT / model_dir_rel
    if not md_abs.exists():
        return None
    runs = sorted([p for p in md_abs.glob(f"{model_name}_*") if (p / "model.pt").exists()])
    return runs[-1] if runs else None

ok = True
for ds in DATASETS:
    md_rel = DATASET_TO_MODELDIR[ds]
    print(f"\n[{ds} -> {md_rel}]")
    for m in MODELS:
        ck = latest_ckpt(m, md_rel)
        if ck is None:
            print(f"  ✗ {m}: NO checkpoint")
            ok = False
        else:
            print(f"  ✓ {m}: {ck.relative_to(REPO_ROOT)}")
if not ok:
    print("\nWARNING: some checkpoints are missing. The sweep will skip those (model, dataset) pairs at runtime.")


## 5) Define the adapted-NETTACK sweep grid


In [ ]:
# Adapted-NETTACK hyperparameter grid. Edit budgets here.
# ATTACK_FRACTIONS is deliberately part of the grid, so target fraction is
# tracked exactly like N_STRUCT / EPS_FEAT in every run's config and summary.
SWEEP = {
    "adapted_nettack": {
        # N_STRUCT is the per-target structural edge-addition budget.
        # EPS_FEAT is the per-target L2 feature budget.
        "N_STRUCT": [1, 2, 5],
        "EPS_FEAT": [0.01, 0.05, 0.1],
        "ATTACK_FRACTION": [float(v) for v in ATTACK_FRACTIONS],
        "D_MIN": [2],
        "CHI2_TAU": [0.004],
        "ENFORCE_DEGREE_CONSTRAINT": [True],
        "SURROGATE_EPOCHS": [200],
        "SURROGATE_LR": [0.01],
        "SURROGATE_WEIGHT_DECAY": [5e-4],
        "CLAMP": [None],
        "VERBOSE": [False],
        "PROGRESS_EVERY": [100],
    },
}

ATTACKS = [ATTACK_TYPE]
print(f"Attack grid for {ATTACK_TYPE!r}: {SWEEP[ATTACK_TYPE]}")


## 6) Run the adapted-NETTACK sweep


In [ ]:
import yaml, importlib.util, os

def write_sweep_yaml(out_path: Path):
    sweeps = []
    for atk in ATTACKS:
        sweeps.append({"attack": atk, "params": SWEEP[atk]})
    cfg = {
        "global": {
            "split": SPLIT,
            "attack_only_illicit": bool(ATTACK_ONLY_ILLICIT),
            "only_clean_correct": bool(ONLY_CLEAN_CORRECT),
            # Default used when ATTACK_FRACTION is not present in a combo.
            "attack_fraction": float(ATTACK_FRACTION),
            "seeds": [int(s) for s in ATTACK_SEEDS],
            "models": list(MODELS),
            "datasets": list(DATASETS),
        },
        "sweeps": sweeps,
    }
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    return out_path

cfg_path = REPO_ROOT / "config" / f"experiments_robustness_adapted_nettack_{DATASET}.yaml"
write_sweep_yaml(cfg_path)
print("Wrote:", cfg_path)

# Run run_sweep.py in-process so the dataset-cache monkey-patches
# from previous cells are visible to the attack drivers. Surrogates are not cached.
_run_sweep_path = REPO_ROOT / "scripts" / "run_sweep.py"
_spec = importlib.util.spec_from_file_location("_run_sweep", str(_run_sweep_path))
_run_sweep_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_run_sweep_mod)

_old_argv = sys.argv
_old_cwd = os.getcwd()
sys.argv = [str(_run_sweep_path), str(cfg_path)]
os.chdir(REPO_ROOT)  # Change working directory to repo root so relative data paths work
try:
    _run_sweep_mod.main()
finally:
    sys.argv = _old_argv
    os.chdir(_old_cwd)


## 7) Load the consolidated results

`run_sweep.py` writes `attacks/results_summary.csv` covering every attack run.


In [ ]:
import pandas as pd, numpy as np

SUMMARY_CSV = REPO_ROOT / "attacks" / "results_summary.csv"
assert SUMMARY_CSV.exists(), "Missing attacks/results_summary.csv"
df = pd.read_csv(SUMMARY_CSV)
print("rows:", len(df), "cols:", len(df.columns))
df.head(2)


## 8) Normalize adapted-NETTACK columns

Static and temporal scripts emit slightly different metric paths. We collapse
both into a tidy frame keyed by `(attack, model, dataset, seed, *budgets)`, with
budget columns for adapted NETTACK including `attack_fraction`.


In [ ]:
def col(df, name):
    return df[name] if name in df.columns else pd.Series([np.nan] * len(df))

def first(*series_list):
    out = None
    for s in series_list:
        if isinstance(s, float) and np.isnan(s):
            continue
        out = s if out is None else out.where(~out.isna(), s)
    return out

out = pd.DataFrame()
out["run_dir"]   = df["run_dir"]
out["attack"]    = col(df, "config.attack").str.lower()
out["model"]     = col(df, "config.model_name")
out["dataset"]   = col(df, "config.dataset")
out["seed"]      = col(df, "config.target_selection.seed")
out["attack_fraction"] = col(df, "config.target_selection.attack_fraction")

# Adapted-NETTACK budgets
out["n_struct"]                  = col(df, "config.attack_params.n_struct")
out["eps_feat"]                  = col(df, "config.attack_params.eps_feat")
out["d_min"]                     = col(df, "config.attack_params.d_min")
out["chi2_tau"]                  = col(df, "config.attack_params.chi2_tau")
out["enforce_degree_constraint"] = col(df, "config.attack_params.enforce_degree_constraint")
out["surrogate_epochs"]          = col(df, "config.attack_params.surrogate_epochs")
out["surrogate_lr"]              = col(df, "config.attack_params.surrogate_lr")
out["surrogate_weight_decay"]    = col(df, "config.attack_params.surrogate_weight_decay")

# Structural accounting when available.
out["nettack_unique_edges_added"] = first(
    col(df, "metrics.attack_effect.structural.n_unique_edges_added"),
    col(df, "metrics.attack_effect.aggregate_concat.n_unique_edges_added_total"),
)
out["nettack_mean_edges_per_target"] = col(df, "metrics.attack_effect.structural.mean_edges_per_target")

# Static metrics layout
f1_pos_clean_s      = col(df, "metrics.classification.aggregate.f1_pos.clean")
f1_pos_adv_s        = col(df, "metrics.classification.aggregate.f1_pos.adv")
f1_pos_drop_s       = col(df, "metrics.classification.aggregate.f1_pos.drop")
recall_pos_clean_s  = col(df, "metrics.classification.aggregate.recall_pos.clean")
recall_pos_adv_s    = col(df, "metrics.classification.aggregate.recall_pos.adv")
f1_macro_clean_s    = col(df, "metrics.classification.aggregate.f1_macro.clean")
f1_macro_adv_s      = col(df, "metrics.classification.aggregate.f1_macro.adv")
roc_clean_s         = col(df, "metrics.classification.aggregate.roc_auc.clean")
roc_adv_s           = col(df, "metrics.classification.aggregate.roc_auc.adv")
asr_s               = col(df, "metrics.attack_effect.asr.value")
asr_pos_s           = col(df, "metrics.attack_effect.asr_pos_neg.asr_pos")
asr_neg_s           = col(df, "metrics.attack_effect.asr_pos_neg.asr_neg")
conf_drop_s         = col(df, "metrics.attack_effect.mean_confidence_drop.value")
attack_time_s       = col(df, "metrics.attack_effect.attack_time_seconds")

# Temporal metrics layout (concat across timesteps)
f1_pos_clean_t      = col(df, "metrics.classification.aggregate_concat.f1_pos_clean")
f1_pos_adv_t        = col(df, "metrics.classification.aggregate_concat.f1_pos_adv")
f1_pos_drop_t       = col(df, "metrics.classification.aggregate_concat.f1_pos_drop")
recall_pos_clean_t  = col(df, "metrics.classification.aggregate_concat.recall_pos_clean")
recall_pos_adv_t    = col(df, "metrics.classification.aggregate_concat.recall_pos_adv")
f1_macro_clean_t    = col(df, "metrics.classification.aggregate_concat.f1_macro_clean")
f1_macro_adv_t      = col(df, "metrics.classification.aggregate_concat.f1_macro_adv")
roc_clean_t         = col(df, "metrics.classification.aggregate_concat.roc_auc_clean")
roc_adv_t           = col(df, "metrics.classification.aggregate_concat.roc_auc_adv")
asr_t               = col(df, "metrics.attack_effect.aggregate_concat.asr")
asr_pos_t           = col(df, "metrics.attack_effect.aggregate_concat.asr_pos")
asr_neg_t           = col(df, "metrics.attack_effect.aggregate_concat.asr_neg")
conf_drop_t         = col(df, "metrics.attack_effect.aggregate_concat.mean_confidence_drop")
attack_time_t       = col(df, "metrics.attack_effect.attack_time_seconds")

out["f1_pos_clean"]     = first(f1_pos_clean_s, f1_pos_clean_t)
out["f1_pos_adv"]       = first(f1_pos_adv_s, f1_pos_adv_t)
out["f1_pos_drop"]      = first(f1_pos_drop_s, f1_pos_drop_t)
out["recall_pos_clean"] = first(recall_pos_clean_s, recall_pos_clean_t)
out["recall_pos_adv"]   = first(recall_pos_adv_s, recall_pos_adv_t)
out["f1_macro_clean"]   = first(f1_macro_clean_s, f1_macro_clean_t)
out["f1_macro_adv"]     = first(f1_macro_adv_s, f1_macro_adv_t)
out["roc_auc_clean"]    = first(roc_clean_s, roc_clean_t)
out["roc_auc_adv"]      = first(roc_adv_s, roc_adv_t)
out["asr"]              = first(asr_s, asr_t)
out["asr_pos"]          = first(asr_pos_s, asr_pos_t)
out["asr_neg"]          = first(asr_neg_s, asr_neg_t)
out["conf_drop"]        = first(conf_drop_s, conf_drop_t)
out["attack_time_s"]    = first(attack_time_s, attack_time_t)

# Normalize attack names across script naming styles.
out["attack"] = out["attack"].str.lower().replace({
    "adaptednettack": "adapted_nettack",
    "adapted_nettack": "adapted_nettack",
})
print(out["attack"].value_counts(dropna=False))
out.head(3)


## 9) Plots — adapted-NETTACK PNG + PDF per (model, dataset)


In [ ]:
import matplotlib.pyplot as plt

def savefig(path_base: Path):
    path_base.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(str(path_base.with_suffix('.png')), dpi=300, bbox_inches='tight')
    plt.savefig(str(path_base.with_suffix('.pdf')), bbox_inches='tight')
    plt.close()

def line_plot(sub, x, y, group_cols, title, xlabel, ylabel, out_base):
    sub = sub.dropna(subset=[x, y]).copy()
    if sub.empty:
        return
    sub[x] = pd.to_numeric(sub[x], errors="coerce")
    sub[y] = pd.to_numeric(sub[y], errors="coerce")
    sub = sub.dropna(subset=[x, y])
    if sub.empty:
        return
    keys = [x] + list(group_cols)
    agg = sub.groupby(keys, dropna=False)[y].agg(['mean', 'std', 'count']).reset_index()
    plt.figure(figsize=(5.5, 3.5))
    if group_cols:
        for gvals, gdf in agg.groupby(list(group_cols), dropna=False):
            if not isinstance(gvals, tuple):
                gvals = (gvals,)
            label = ", ".join(f"{c}={v}" for c, v in zip(group_cols, gvals))
            gdf = gdf.sort_values(x)
            plt.plot(gdf[x], gdf['mean'], marker='o', label=label)
            if (gdf['count'] > 1).any():
                plt.fill_between(gdf[x],
                                 gdf['mean'] - gdf['std'].fillna(0),
                                 gdf['mean'] + gdf['std'].fillna(0),
                                 alpha=0.2)
        plt.legend(fontsize=8)
    else:
        agg = agg.sort_values(x)
        plt.plot(agg[x], agg['mean'], marker='o')
        if (agg['count'] > 1).any():
            plt.fill_between(agg[x],
                             agg['mean'] - agg['std'].fillna(0),
                             agg['mean'] + agg['std'].fillna(0),
                             alpha=0.2)
    plt.title(title, fontsize=10)
    plt.xlabel(xlabel); plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    savefig(out_base)

ART = REPO_ROOT / ARTIFACTS_DIR
ART.mkdir(parents=True, exist_ok=True)

for (ds, model), sub in out.groupby(["dataset", "model"]):
    base = ART / ds / model / "plots"
    nettack = sub[sub["attack"] == "adapted_nettack"]

    line_plot(nettack, "eps_feat", "asr", ["n_struct", "attack_fraction"],
              f"Adapted-NETTACK | {model} on {ds} - ASR vs eps_feat",
              "eps_feat", "ASR",
              base / "nettack_asr_vs_epsfeat")
    line_plot(nettack, "eps_feat", "f1_pos_adv", ["n_struct", "attack_fraction"],
              f"Adapted-NETTACK | {model} on {ds} - F1_pos vs eps_feat",
              "eps_feat", "F1_pos (adv)",
              base / "nettack_f1pos_vs_epsfeat")
    line_plot(nettack, "n_struct", "asr", ["eps_feat", "attack_fraction"],
              f"Adapted-NETTACK | {model} on {ds} - ASR vs n_struct",
              "n_struct", "ASR",
              base / "nettack_asr_vs_nstruct")

print("Plots written under:", ART)


## 10) Tables — adapted-NETTACK CSV + LaTeX


In [ ]:
def to_latex(df_, path: Path, caption: str, label: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    tex = df_.to_latex(
        index=False,
        float_format=lambda x: f"{x:.4f}" if isinstance(x, (float, np.floating)) else str(x),
        escape=False,
    )
    insert = "\\toprule\n" + f"\\caption{{{caption}}}\\label{{{label}}}\\\\\n"
    tex = tex.replace("\\toprule", insert, 1)
    path.write_text(tex, encoding="utf-8")

agg_cols = {
    "f1_pos_clean": "mean", "f1_pos_adv": "mean", "f1_pos_drop": "mean",
    "recall_pos_clean": "mean", "recall_pos_adv": "mean",
    "f1_macro_clean": "mean", "f1_macro_adv": "mean",
    "roc_auc_clean": "mean", "roc_auc_adv": "mean",
    "asr": "mean", "asr_pos": "mean", "asr_neg": "mean",
    "conf_drop": "mean", "attack_time_s": "mean",
    "nettack_unique_edges_added": "mean", "nettack_mean_edges_per_target": "mean",
}
agg_cols = {k: v for k, v in agg_cols.items() if k in out.columns}

ATTACK_TABLE_KEYS = {
    "adapted_nettack": [
        "eps_feat", "n_struct", "attack_fraction",
        "d_min", "chi2_tau", "enforce_degree_constraint",
    ],
}

# 1) Headline robustness summary per (dataset, model, adapted-NETTACK budgets)
for ds, sub_ds in out.groupby("dataset"):
    for model, sub in sub_ds.groupby("model"):
        tdir = ART / ds / model / "tables"
        tdir.mkdir(parents=True, exist_ok=True)

        atk_grp = sub[sub["attack"] == "adapted_nettack"]
        keys = [k for k in ATTACK_TABLE_KEYS["adapted_nettack"] if k in atk_grp.columns]
        if keys and not atk_grp.empty:
            grp = (atk_grp.groupby(keys, dropna=False)
                          .agg(agg_cols).reset_index()
                          .sort_values(keys))
            csv_path = tdir / "adapted_nettack_summary.csv"
            grp.to_csv(csv_path, index=False)
            to_latex(
                grp, tdir / "adapted_nettack_summary.tex",
                caption=f"Adapted-NETTACK robustness - {model} on {ds} (mean over seeds)",
                label=f"tab:{ds}:{model}:adapted_nettack",
            )

        base = sub.groupby("attack")[["f1_pos_clean", "recall_pos_clean", "f1_macro_clean", "roc_auc_clean"]].mean().reset_index()
        base.to_csv(tdir / "clean_baseline.csv", index=False)

# 2) Cross-model comparison per dataset
for ds, sub in out.groupby("dataset"):
    sub = sub[sub["attack"] == "adapted_nettack"]
    keys = ["model"] + ATTACK_TABLE_KEYS["adapted_nettack"]
    keys = [k for k in keys if k in sub.columns]

    if keys and not sub.empty:
        grp = (sub.groupby(keys, dropna=False)
                  .agg(agg_cols).reset_index()
                  .sort_values(keys))
        out_dir = ART / ds / "_cross_model" / "tables"
        out_dir.mkdir(parents=True, exist_ok=True)
        grp.to_csv(out_dir / "adapted_nettack_cross_model.csv", index=False)
        to_latex(grp, out_dir / "adapted_nettack_cross_model.tex",
                 caption=f"Adapted-NETTACK cross-model - {ds}",
                 label=f"tab:{ds}:cross_model:adapted_nettack")

print("Tables written under:", ART)


## 10b) Final paper-ready summary CSV — adapted-NETTACK × dataset

Filters the full sweep to `attack == adapted_nettack` and `dataset == DATASET`,
aggregates over seeds, and writes a single self-contained CSV. `ATTACK_FRACTION`
is included as a budget column.


In [ ]:
# Build the requested adapted-NETTACK summary CSV.
sub = out[(out["attack"] == "adapted_nettack") & (out["dataset"] == DATASET)].copy()

budget_keys = [
    "eps_feat", "n_struct", "attack_fraction",
    "d_min", "chi2_tau", "enforce_degree_constraint",
    "surrogate_epochs", "surrogate_lr", "surrogate_weight_decay",
]
budget_keys = [k for k in budget_keys if k in sub.columns]

metric_cols = [
    "f1_pos_clean", "f1_pos_adv", "f1_pos_drop",
    "recall_pos_clean", "recall_pos_adv",
    "f1_macro_clean", "f1_macro_adv",
    "roc_auc_clean", "roc_auc_adv",
    "asr", "asr_pos", "asr_neg",
    "conf_drop", "attack_time_s",
    "nettack_unique_edges_added", "nettack_mean_edges_per_target",
]
metric_cols = [c for c in metric_cols if c in sub.columns]

# Drop all-empty metrics to avoid sparse, noisy summaries.
metric_cols = [c for c in metric_cols if not sub[c].isna().all()]

group_cols = ["attack", "dataset", "model"] + budget_keys
group_cols = [c for c in group_cols if c in sub.columns]

if sub.empty:
    print(f"No rows for attack=adapted_nettack, dataset={DATASET} in the summary.")
    summary = sub
else:
    agg = sub.groupby(group_cols, dropna=False)[metric_cols].agg(["mean", "std", "count"])
    agg.columns = [f"{m}_{stat}" for m, stat in agg.columns]
    summary = agg.reset_index().sort_values(group_cols)

summary_dir = ART / DATASET / "_summary"
summary_dir.mkdir(parents=True, exist_ok=True)
summary_path = summary_dir / f"summary_adapted_nettack_{DATASET}.csv"
summary.to_csv(summary_path, index=False)
print(f"Wrote: {summary_path}  (rows={len(summary)})")
summary.head(20)


## 11) Export artifacts (zip + download)

In [ ]:
import zipfile

zip_path = REPO_ROOT / f"{ARTIFACTS_DIR}.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in (REPO_ROOT / ARTIFACTS_DIR).rglob("*"):
        if p.is_file():
            z.write(p, arcname=str(p.relative_to(REPO_ROOT)))
    if SUMMARY_CSV.exists():
        z.write(SUMMARY_CSV, arcname="attacks/results_summary.csv")

print("Wrote:", zip_path)

try:
    from google.colab import files  # type: ignore
    files.download(str(zip_path))
except Exception:
    print("Not on Colab — find the zip at:", zip_path)
